# Datawhale 学习项目｜《Hello-Agents》第四章学习笔记 + 代码实践

> 基于 Datawhale《Hello-Agents》V1.0.3（2026-07-17 PDF）第四章《智能体经典范式构建》整理。

## 使用说明

- 默认代码单元只依赖 **Python 标准库**，可离线运行。
- 真实 API 客户端以 `RUN_REAL_API = False` 开关保留，默认不执行。
- 本章复现三种经典范式：**ReAct**、**Plan-and-Solve**、**Reflection**，并补充思考题参考作答。

## 目录

1. 环境准备与基础组件：LLM 客户端、工具执行器
2. ReAct：思考 - 行动 - 观察循环
3. Plan-and-Solve：先规划，后执行
4. Reflection：执行 - 反思 - 优化
5. 三种范式对比与思考题参考作答

---

# 4.1 环境准备与基础工具定义

## 核心笔记

教材建议先安装：

```bash
pip install openai python-dotenv
```

并在项目根目录创建 `.env`：

```text
LLM_API_KEY="YOUR-API-KEY"
LLM_MODEL_ID="YOUR-MODEL"
LLM_BASE_URL="YOUR-URL"
```

### `HelloAgentsLLM`

教材封装了一个兼容 OpenAI 接口的客户端类，核心方法 `think(messages, temperature=0)`：

- 从环境变量或 `.env` 读取 `model / apiKey / baseUrl / timeout`
- 调用 `client.chat.completions.create(..., stream=True)`
- 逐块收集流式输出，拼接成完整字符串返回

这样做的好处是：**主逻辑只关心“给模型什么消息、拿回什么文本”，不与 SDK 细节耦合。**

### 工具的三要素

一个良好定义的工具包含：

1. **Name**：唯一标识符，如 `Search`
2. **Description**：自然语言描述，LLM 靠它判断何时使用
3. **Execution Logic**：真正执行的函数

`ToolExecutor` 负责注册工具、按名称取工具、生成可用工具描述字符串。

In [ ]:
# 必做代码 4.1：工具执行器 ToolExecutor（教材 4.2.2 的通用管理器）
from typing import Dict, Any, Callable


class ToolExecutor:
    """管理工具的注册、查询与描述生成。"""

    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: Callable):
        if name in self.tools:
            print(f"警告: 工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> Callable:
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        return "\n".join(
            f"- {name}: {info['description']}" for name, info in self.tools.items()
        )


# 演示：注册两个本地可运行工具，无需联网
def mock_search(query: str) -> str:
    """离线模拟搜索：按关键词返回预置事实。"""
    print(f"🔍 正在执行 [MockSearch] 网页搜索: {query}")
    if "华为" in query or "手机" in query:
        return (
            "[1] 华为手机官网: Mate 系列、Pura 系列、Pocket 系列、nova 系列。\n"
            "[2] 2025 年推荐: Mate 70 与 Pura 80 Pro+ 为近期旗舰。\n"
            "[3] 卖点: Mate 70 顶级拍照、全焦段覆盖；Pura 80 Pro+ 先锋影像。"
        )
    return f"未找到关于 '{query}' 的模拟结果。"


def calculator(expression: str) -> str:
    """安全计算简单算术表达式（仅支持数字、括号和 + - * /）。"""
    import ast

    allowed = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Num,
               ast.Add, ast.Sub, ast.Mult, ast.Div, ast.USub, ast.UAdd)

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Num):
            return node.n
        if isinstance(node, ast.BinOp):
            left, right = _eval(node.left), _eval(node.right)
            if isinstance(node.op, ast.Add):
                return left + right
            if isinstance(node.op, ast.Sub):
                return left - right
            if isinstance(node.op, ast.Mult):
                return left * right
            if isinstance(node.op, ast.Div):
                return left / right
        if isinstance(node, ast.UnaryOp):
            value = _eval(node.operand)
            if isinstance(node.op, ast.USub):
                return -value
            if isinstance(node.op, ast.UAdd):
                return value
        raise ValueError("表达式中包含不支持的语法")

    tree = ast.parse(expression.strip(), mode="eval")
    for node in ast.walk(tree):
        if not isinstance(node, allowed):
            raise ValueError("表达式中包含不支持的语法")
    result = _eval(tree)
    return str(round(result, 4))


toolExecutor = ToolExecutor()
toolExecutor.registerTool(
    "Search",
    "一个网页搜索引擎。当需要回答时事、事实或知识库中没有的信息时使用。",
    mock_search,
)
toolExecutor.registerTool(
    "Calculator",
    "一个精确计算器。当需要计算数学表达式时使用，输入为合法的算术表达式。",
    calculator,
)

print("\n--- 可用工具 ---")
print(toolExecutor.getAvailableTools())
print("\n--- 测试 Calculator ---")
print("(123 + 456) * 789 / 12 =", toolExecutor.getTool("Calculator")("(123 + 456) * 789 / 12"))

In [ ]:
# 可选进阶 4.2：教材真实 LLM 客户端（默认不执行）
# 启用后需安装 openai / python-dotenv，并配置 .env 或环境变量。
RUN_REAL_API = False

if RUN_REAL_API:
    import os
    from typing import List, Dict
    from openai import OpenAI
    from dotenv import load_dotenv

    load_dotenv()

    class HelloAgentsLLM:
        """调用任意兼容 OpenAI 接口的服务，默认流式响应。"""

        def __init__(self, model=None, apiKey=None, baseUrl=None, timeout=None):
            self.model = model or os.getenv("LLM_MODEL_ID")
            apiKey = apiKey or os.getenv("LLM_API_KEY")
            baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
            timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
            if not all([self.model, apiKey, baseUrl]):
                raise ValueError("模型 ID、API 密钥和服务地址必须提供。")
            self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

        def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            collected = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                collected.append(content)
            return "".join(collected)

    print("HelloAgentsLLM 已就绪。")
else:
    print("未启用真实 API，后续使用 MockLLM 保证学习项目可离线运行。")

---

# 4.2 ReAct：Reasoning and Acting

## 核心笔记

ReAct（Yao et al., 2022）把 **推理** 和 **行动** 显式结合成循环：

```text
Thought → Action → Observation → Thought → ...
```

- **Thought**：分析当前情况、拆解任务、制定下一步计划或反思上一步结果。
- **Action**：调用外部工具，格式如 `Search[华为最新手机]`。
- **Observation**：工具返回结果，追加进历史，作为下一轮输入。

它解决了两类旧范式的不足：

- 纯思考（CoT）：能推理但不能接触外部世界，容易事实幻觉。
- 纯行动：缺少规划与纠错能力。

适用场景：需要外部知识、精确计算、API 交互的任务。

### ReAct 的特点与局限

优点：

- 高可解释性：Thought 链展示“心路历程”
- 动态规划与纠错：走一步看一步，根据 Observation 调整
- 工具协同：LLM 负责推理，工具负责执行

局限：

- 强依赖 LLM 的推理、指令遵循和格式化输出能力
- 多轮串行调用导致延迟与成本较高
- 提示词模板脆弱，微小改动可能影响行为
- 缺乏全局规划，可能陷入局部最优或原地打转

调试技巧：打印完整提示词、打印原始输出、验证工具输入输出、加 few-shot 示例、调低 temperature。

In [ ]:
# 必做代码 4.3：离线 MockLLM —— 模拟教材中 LLM 的 Thought/Action 输出
import re


class MockLLM:
    """根据提示词中的关键词，按教材格式生成下一步输出。"""

    def __init__(self):
        self.reflect_stage = 0

    def think(self, messages, temperature: float = 0) -> str:
        prompt = messages[-1]["content"] if isinstance(messages, list) else str(messages)
        print("🧠 正在调用 MockLLM ...")

        # ---------- ReAct ----------
        if "可用工具如下" in prompt:
            if "Observation:" not in prompt:
                if "华为" in prompt:
                    return (
                        "Thought: 要回答这个问题，需要查询最新手机型号和卖点，知识库可能过时。\n"
                        "Action: Search[华为最新手机型号及主要卖点]"
                    )
                if "123" in prompt or "计算" in prompt:
                    return (
                        "Thought: 这是精确计算题，应交给计算器而不是心算。\n"
                        "Action: Calculator[(123 + 456) * 789 / 12]"
                    )
                return (
                    "Thought: 需要外部信息，先使用搜索工具。\n"
                    f"Action: Search[{prompt[:20]}]"
                )

            observation = re.findall(r"Observation:\s*(.*)", prompt)[-1]
            if "Calculator[" in prompt:
                return (
                    f"Thought: 工具返回 {observation}，可以给出最终答案。\n"
                    f"Action: Finish[计算结果为 {observation}]"
                )
            return (
                "Thought: 搜索结果提供了型号和卖点，可以总结回答。\n"
                "Action: Finish[华为近期旗舰包括 Mate 70 与 Pura 80 Pro+；"
                "Mate 70 主打顶级拍照和全焦段覆盖，Pura 80 Pro+ 强调先锋影像。]"
            )

        # ---------- Plan-and-Solve: Planner ----------
        if "AI 规划专家" in prompt:
            return (
                "```python\n"
                "[\n"
                '  "计算周一卖出的苹果数量：15 个",\n'
                '  "计算周二卖出的苹果数量：周一数量 × 2 = 15 × 2 = 30 个",\n'
                '  "计算周三卖出的苹果数量：周二数量 - 5 = 30 - 5 = 25 个",\n'
                '  "计算三天总销量：15 + 30 + 25 = 70 个"\n'
                "]\n"
                "```"
            )

        # ---------- Plan-and-Solve: Executor ----------
        if "AI 执行专家" in prompt:
            step_match = re.search(r"# 当前步骤:\s*(.*)", prompt)
            step = step_match.group(1).strip() if step_match else prompt
            if "总销量" in step:
                return "70"
            if "周三" in step:
                return "25"
            if "周二" in step:
                return "30"
            if "周一" in step:
                return "15"
            return "0"

        # ---------- Reflection ----------
        # 注意：优化提示词里也会出现“代码评审专家”，必须先判断“你正在根据”
        if "你正在根据" in prompt:
            return MOCK_SIEVE_CODE

        if "代码评审专家" in prompt:
            if "Eratosthenes" in prompt or "is_prime = [True]" in prompt:
                return (
                    "当前代码使用 Eratosthenes 筛法，时间复杂度 O(n log log n)，"
                    "空间复杂度 O(n)，一般场景下无需改进。"
                )
            return (
                "当前代码对每个数做试除法，时间复杂度 O(n * sqrt(n))。"
                "建议使用埃拉托斯特尼筛法（Sieve of Eratosthenes），"
                "时间复杂度 O(n log log n)，显著提升性能。"
            )

        if "资深 Python 程序员" in prompt:
            return MOCK_NAIVE_CODE

        return "MockLLM: 未识别的提示词类型。"

    # 让类属性在方法定义后仍可用
    MOCK_NAIVE_CODE = ""
    MOCK_SIEVE_CODE = ""


MOCK_NAIVE_CODE = """def find_primes(n):
    # Return all prime numbers between 1 and n using trial division.
    if n < 2:
        return []
    primes = []
    for num in range(2, n + 1):
        is_prime = True
        for i in range(2, int(num ** 0.5) + 1):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return primes
"""

MOCK_SIEVE_CODE = """def find_primes(n):
    # Find primes up to n using the Sieve of Eratosthenes.
    if n < 2:
        return []
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    p = 2
    while p * p <= n:
        if is_prime[p]:
            for i in range(p * p, n + 1, p):
                is_prime[i] = False
        p += 1
    return [num for num in range(2, n + 1) if is_prime[num]]
"""

MockLLM.MOCK_NAIVE_CODE = MOCK_NAIVE_CODE
MockLLM.MOCK_SIEVE_CODE = MOCK_SIEVE_CODE

print("MockLLM 已定义，支持 ReAct / Plan-and-Solve / Reflection 三类提示词。")

In [ ]:
# 必做代码 4.4：完整 ReActAgent（教材 4.2.3 的离线可运行版）
import re

REACT_PROMPT_TEMPLATE = """
你是一个有能力调用外部工具的智能助手。
可用工具如下:
{tools}
请严格按照以下格式进行回应:
Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`: 调用一个可用工具。
- `Finish[最终答案]`: 当你认为已经获得最终答案时。
当你收集到足够信息时，必须在 Action 后使用 Finish[最终答案]。
现在，请开始解决以下问题:
Question: {question}
History: {history}
"""


class ReActAgent:
    def __init__(self, llm_client, tool_executor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def _parse_output(self, text: str):
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None

    def run(self, question: str):
        self.history = []
        print(f"\n--- 开始处理问题: {question} ---")

        for step in range(1, self.max_steps + 1):
            print(f"\n--- 第 {step} 步 ---")
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=self.tool_executor.getAvailableTools(),
                question=question,
                history="\n".join(self.history),
            )
            response_text = self.llm_client.think([{"role": "user", "content": prompt}])
            if not response_text:
                print("错误: LLM 未能返回有效响应。")
                break

            thought, action = self._parse_output(response_text)
            if thought:
                print(f"🤔 思考: {thought}")
            if not action:
                print("警告: 未能解析出有效 Action，流程终止。")
                break

            if action.startswith("Finish"):
                final_answer = re.match(r"Finish\[(.*)\]", action, re.DOTALL).group(1)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer

            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                print("警告: Action 格式无效。")
                continue

            print(f"🎬 行动: {tool_name}[{tool_input}]")
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误: 未找到名为 '{tool_name}' 的工具。"
            else:
                observation = tool_function(tool_input)

            print(f"👀 观察: {observation}")
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        print("已达到最大步数，流程终止。")
        return None


react_agent = ReActAgent(MockLLM(), toolExecutor, max_steps=5)
react_agent.run("华为最新的手机是哪一款？它的主要卖点是什么？")
print("\n" + "=" * 55)
react_agent.run("计算 (123 + 456) * 789 / 12 = ? 的结果")

In [ ]:
# 必做代码 4.5：更鲁棒的 JSON 输出解析方案（对比正则表达式）
import json
import re


def parse_react_output_json(text: str):
    """从模型的 JSON 输出中解析 thought 与 action。"""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("未找到 JSON 对象")
    data = json.loads(match.group(0))
    return data.get("thought"), data.get("action")


json_output = """
这里是模型输出前的一些说明文字。
{
  "thought": "需要精确计算，应调用计算器。",
  "action": "Calculator[(123 + 456) * 789 / 12]"
}
"""
thought, action = parse_react_output_json(json_output)
print("解析结果:")
print("  Thought:", thought)
print("  Action :", action)

print("\n对比:")
print("  正则解析: 轻量、兼容自由文本，但格式漂移时容易失败。")
print("  JSON 解析: 结构明确、字段校验方便，但要求模型严格输出合法 JSON。")
print("  工程上可组合: 优先 JSON 解析，失败后回退到正则，并记录原始输出便于调试。")

---

# 4.3 Plan-and-Solve：先规划，后执行

## 核心笔记

Plan-and-Solve Prompting（Wang et al., 2023）把任务拆成两个阶段：

1. **规划阶段 Planning**：LLM 接收原始问题，一次性生成结构化计划  
   $$P = \pi_{plan}(q) = (p_1, p_2, ..., p_n)$$
2. **执行阶段 Solving**：按计划逐步执行，每一步都依赖原始问题、完整计划和之前结果  
   $$s_i = \pi_{solve}(q, P, (s_1, ..., s_{i-1}))$$

最终答案是最后一步的结果 $s_n$。

### 与 ReAct 的对比

| 维度 | ReAct | Plan-and-Solve |
|---|---|---|
| 思考方式 | 边想边做，动态调整 | 先谋后动，蓝图固定 |
| 上下文 | 累积 Action/Observation 历史 | 累积每步执行结果 |
| 优势 | 环境适应、动态纠错 | 结构清晰、目标一致 |
| 劣势 | 可能短视、原地打转 | 计划静态，执行中难以修正 |
| 适用 | 搜索、API、实时信息 | 多步数学、报告撰写、代码生成 |

教材强调：规划提示词要求输出 **Python 列表**，并用 `ast.literal_eval` 安全解析，比解析自然语言更稳定。

In [ ]:
# 必做代码 4.6：Planner + Executor + PlanAndSolveAgent（离线可运行版）
import ast

PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的 AI 规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个 Python 列表，其中每个元素都是一个描述子任务的字符串。
问题: {question}
请严格按照以下格式输出你的计划:
```python
["步骤 1", "步骤 2", "步骤 3", ...]
```
"""

EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的 AI 执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请专注于解决“当前步骤”，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。
# 原始问题:
{question}
# 完整计划:
{plan}
# 历史步骤与结果:
{history}
# 当前步骤:
{current_step}
请仅输出针对“当前步骤”的回答:
"""


class Planner:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def plan(self, question: str) -> list:
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)
        print("--- 正在生成计划 ---")
        response_text = self.llm_client.think([{"role": "user", "content": prompt}]) or ""
        print(f"✅ 计划已生成:\n{response_text}")
        try:
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"❌ 解析计划时出错: {e}")
            return []


class Executor:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def execute(self, question: str, plan: list) -> str:
        history = ""
        print("\n--- 正在执行计划 ---")
        for i, step in enumerate(plan):
            print(f"\n-> 正在执行步骤 {i + 1}/{len(plan)}: {step}")
            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "无",
                current_step=step,
            )
            response_text = self.llm_client.think([{"role": "user", "content": prompt}]) or ""
            history += f"步骤 {i + 1}: {step}\n结果: {response_text}\n\n"
            print(f"✅ 步骤 {i + 1} 已完成，结果: {response_text}")
        return response_text


class PlanAndSolveAgent:
    def __init__(self, llm_client):
        self.planner = Planner(llm_client)
        self.executor = Executor(llm_client)

    def run(self, question: str):
        print(f"\n--- 开始处理问题 ---\n问题: {question}")
        plan = self.planner.plan(question)
        if not plan:
            print("任务终止: 无法生成有效计划。")
            return None
        final_answer = self.executor.execute(question, plan)
        print(f"\n--- 任务完成 ---\n最终答案: {final_answer}")
        return final_answer


question = (
    "一个水果店周一卖出了 15 个苹果。周二卖出的苹果数量是周一的两倍。"
    "周三卖出的数量比周二少了 5 个。请问这三天总共卖出了多少个苹果？"
)
PlanAndSolveAgent(MockLLM()).run(question)

### 动态重规划思路（教材习题 4 的延伸）

教材的实现中计划是静态的。若某一步失败或结果异常，可以引入“执行校验 + 重规划”：

```text
生成计划 P
for step in P:
    result = execute(step)
    if validate(result) 失败:
        feedback = diagnose(step, result)
        P = replan(question, P, feedback)   # 从失败步骤开始重规划
        continue
    记录 result
```

这相当于把 Plan-and-Solve 与 ReAct 的纠错能力结合起来：保留全局蓝图，同时允许局部修正。

---

# 4.4 Reflection：执行 - 反思 - 优化

## 核心笔记

Reflection 为智能体引入**事后自我校正循环**，灵感来自人类“写初稿 → 校对 → 修订”。

三步循环：

1. **执行 Execution**：生成初稿（可用 ReAct / Plan-and-Solve 或其他方式）。
2. **反思 Reflection**：由“评审员”模型从事实性、逻辑漏洞、效率、遗漏等维度给出结构化反馈。
3. **优化 Refinement**：把初稿 + 反馈作为新上下文，生成修订稿。

形式化：

$$F_i = \pi_{reflect}(Task, O_i)$$

$$O_{i+1} = \pi_{refine}(Task, O_i, F_i)$$

循环直到反馈为“无需改进”或达到最大迭代次数。

### 价值

- 不依赖外部工具反馈，能修正更高层次的逻辑与策略错误
- 把一次性执行变成持续优化
- 形成“短期记忆”：记录如何从有缺陷初稿迭代到最终版本

### 成本收益

成本：每轮至少多两次模型调用，延迟显著增加，提示词设计更复杂。  
收益：方案质量跃迁，鲁棒性与可靠性增强。

适合：关键业务代码、技术报告、复杂推演、决策支持。  
不适合：实时响应或“大致正确即可”的场景。

In [ ]:
# 必做代码 4.7：Memory 短期记忆模块（教材 4.4.2）
from typing import List, Dict, Any, Optional


class Memory:
    """存储每次执行与反思的轨迹。"""

    def __init__(self):
        self.records: List[Dict[str, Any]] = []

    def add_record(self, record_type: str, content: str):
        self.records.append({"type": record_type, "content": content})
        print(f"📝 记忆已更新，新增一条 '{record_type}' 记录。")

    def get_trajectory(self) -> str:
        parts = []
        for record in self.records:
            if record["type"] == "execution":
                parts.append(f"--- 上一轮尝试(代码) ---\n{record['content']}")
            elif record["type"] == "reflection":
                parts.append(f"--- 评审员反馈 ---\n{record['content']}")
        return "\n\n".join(parts)

    def get_last_execution(self) -> Optional[str]:
        for record in reversed(self.records):
            if record["type"] == "execution":
                return record["content"]
        return None


memory = Memory()
memory.add_record("execution", "def f(x): return x")
memory.add_record("reflection", "建议增加参数校验。")
print(memory.get_trajectory())
print("\n最近一次执行:\n", memory.get_last_execution())

In [ ]:
# 必做代码 4.8：ReflectionAgent 完整循环（教材 4.4.3 的离线可运行版）
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的 Python 程序员。请根据以下要求，编写一个 Python 函数。
你的代码必须包含完整的函数签名、文档字符串，并遵循 PEP 8 编码规范。
要求: {task}
请直接输出代码，不要包含任何额外的解释。
"""

REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下 Python 代码，并专注于找出其在算法效率上的主要瓶颈。
# 原始任务:
{task}
# 待审查的代码:
```python
{code}
```
请分析该代码的时间复杂度，并提出算法上更优的改进建议。
如果代码在算法层面已经达到最优，才能回答“无需改进”。
"""

REFINE_PROMPT_TEMPLATE = """
你是一位资深的 Python 程序员。你正在根据一位代码评审专家的反馈来优化你的代码。
# 原始任务:
{task}
# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈:
{feedback}
请根据反馈生成优化后的新版本代码，直接输出代码。
"""


class ReflectionAgent:
    def __init__(self, llm_client, max_iterations: int = 3):
        self.llm_client = llm_client
        self.memory = Memory()
        self.max_iterations = max_iterations

    def _respond(self, prompt: str) -> str:
        return self.llm_client.think([{"role": "user", "content": prompt}]) or ""

    def run(self, task: str) -> str:
        print(f"\n--- 开始处理任务 ---\n任务: {task}")

        print("\n--- 正在进行初始尝试 ---")
        initial_code = self._respond(INITIAL_PROMPT_TEMPLATE.format(task=task))
        self.memory.add_record("execution", initial_code)

        for i in range(self.max_iterations):
            print(f"\n--- 第 {i + 1}/{self.max_iterations} 轮迭代 ---")

            print("\n-> 正在进行反思 ...")
            last_code = self.memory.get_last_execution()
            feedback = self._respond(REFLECT_PROMPT_TEMPLATE.format(task=task, code=last_code))
            self.memory.add_record("reflection", feedback)
            print(feedback)

            if "无需改进" in feedback:
                print("\n✅ 反思认为代码已无需改进，任务完成。")
                break

            print("\n-> 正在进行优化 ...")
            refined_code = self._respond(REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback,
            ))
            self.memory.add_record("execution", refined_code)

        final_code = self.memory.get_last_execution()
        print(f"\n--- 任务完成 ---\n最终代码:\n```python\n{final_code}\n```")
        return final_code


task = "编写一个 Python 函数，找出 1 到 n 之间所有的素数 (prime numbers)。"
agent = ReflectionAgent(MockLLM(), max_iterations=3)
final_code = agent.run(task)

# 验证最终代码确实可执行
namespace = {}
exec(final_code, namespace)
print("\n验证 find_primes(30) =", namespace["find_primes"](30))

---

# 4.5 三种范式对比与选择

| 范式 | 思考与行动的组织方式 | 核心优势 | 主要代价 | 典型场景 |
|---|---|---|---|---|
| **ReAct** | 每一步都 Thought + Action + Observation，边想边做 | 动态纠错、可解释、善用工具 | 多轮调用成本高；可能短视或循环 | 搜索、实时信息、API 操作 |
| **Plan-and-Solve** | 先一次性规划，再逐步执行 | 结构清晰、目标一致、稳定 | 计划静态，执行中不易修正 | 多步数学、报告、代码结构生成 |
| **Reflection** | 执行 → 反思 → 优化，事后迭代 | 质量跃迁、自我纠错 | 成本与延迟成倍增加 | 关键代码、报告、决策支持 |

实践中的组合方式：

```text
Plan-and-Solve 做顶层蓝图
    ↓
每个步骤内部用 ReAct 调工具获取事实
    ↓
全部完成后用 Reflection 做质量检查与修订
```

这就是一个兼顾**全局结构、环境交互和结果质量**的混合范式。

# 思考题参考作答

## 1. 三种范式的本质区别？智能家居助手选哪种？

**本质区别：**

- **ReAct**：思考和行动交织在每一步，下一轮决策依赖上一轮 Observation，是“走一步看一步”。
- **Plan-and-Solve**：思考集中在前，行动集中在后，计划一次性生成且通常不再修改，是“先谋后动”。
- **Reflection**：执行完成后再增加一轮“审视结果 → 给出反馈 → 修订”的回路，是“做完后返工优化”。

**智能家居控制助手**：建议以 **ReAct 为基础架构**。原因是环境状态持续变化（人是否在家、光线、温度、用户习惯），设备控制需要根据实时反馈动态调整；ReAct 的 Observation 闭环天然适合“感知-决策-执行-再感知”。可以叠加 Reflection：每天夜间复盘当天控制效果，优化用户习惯模型；对固定场景（如“离家模式”）用 Plan-and-Solve 生成标准流程。

**混合架构**：Planner 生成高层计划（如“准备晚餐场景”）→ ReAct 逐步调用设备 API 并观察状态 → Reflection 在结束后评估舒适度与能耗并修正策略。

## 2. 正则解析的脆弱性与更鲁棒方案

正则脆弱性：

- 模型输出多一段解释或换行位置变化，`Action:` 可能匹配失败。
- 工具输入本身含 `[`、`]`、`Thought:` 等字符时，`(\w+)\[(.*)\]` 容易贪婪错配。
- 模型先输出 Finish 又补充一句说明，可能截断错误。
- 中英文标点、全角冒号、多余空格都会导致不匹配。

更鲁棒方案：

1. **JSON / YAML 结构化输出**：用 `json.loads` 解析字段，配合 schema 校验。
2. **工具调用原生 API**：使用 OpenAI function calling / tool calling，直接拿到结构化参数。
3. **多重回退解析**：先 JSON，失败回退正则，再失败把原始输出喂给模型让其自修正。
4. **输出校验器**：检查工具名是否在注册表中、参数类型是否符合函数签名。

优缺点：正则简单零依赖但脆弱；JSON 清晰可校验但要求模型严格遵循；原生 function calling 最稳但与具体模型绑定。

## 3. ToolExecutor 扩展：计算器、失败纠正与工具检索

**计算器**：注册 `Calculator` 工具，用 `ast` 白名单解析表达式，避免直接 `eval` 执行任意代码。

**工具选择失败处理**：

- 记录连续失败次数；
- 把错误信息（工具不存在 / 参数类型错误）作为 Observation 返回给 LLM；
- 提示中追加：`上次工具调用失败，请检查工具名和参数格式后重试`；
- 超过阈值则转入人工或降级流程。

**工具数量扩展到 50~100 个**：扁平描述列表会超出上下文且干扰注意力。优化方式：

- 按类别分组，先让模型选类别，再在类别内选工具；
- 用向量检索根据 Action 意图召回 Top-k 工具描述，动态注入提示词；
- 给工具加标签、示例调用和参数 schema；
- 高频工具常驻提示词，长尾工具按需检索。

## 4. 动态重规划、商务旅行范式选择与分层规划

**动态重规划**：在执行每步后加入校验器；若失败或结果异常，把“失败步骤 + 错误原因 + 已完成结果”反馈给 Planner，让它从该步开始重新生成剩余计划，而不是从头再来。

**北京→上海商务旅行**：更适合 **Plan-and-Solve 为主 + ReAct 为辅**。机票、酒店、租车有明确依赖顺序（先定行程再订酒店），适合先规划；但实时票价和余量需要 ReAct 调 API 获取并动态修正。

**分层规划**：先生成抽象计划（出行 → 交通 → 住宿 → 租车），再为每个抽象步骤生成子计划。优势是降低单次规划难度、便于局部重规划、提高可读性和可维护性。

## 5. 双模型 Reflection、终止条件与多维度反思

**双模型**：用快模型做执行、强模型做反思，可提升批判质量且控制成本；但两者“认知水平”不一致可能导致反馈过严或执行模型无法理解反馈。需要在提示词中明确反馈的可操作粒度。

**更智能的终止条件**：

- 反馈置信度低于阈值即停止；
- 连续两轮无实质性改进（用文本相似度或评分衡量）即停止；
- 引入外部验证器，通过测试即停止；
- 设置预算上限（调用次数 / token / 时间）。

**学术论文写作助手的多维 Reflection**：分别设置“逻辑评审员”“方法评审员”“语言评审员”“引用规范评审员”，各自输出结构化评分与建议；聚合后按优先级修订，直到各维度评分达标。

## 6. 提示词差异、角色设定与 few-shot

**ReAct 提示词**强调“每步输出 Thought/Action、工具格式、Finish 条件”，服务于动态循环；**Plan-and-Solve 提示词**强调“输出 Python 列表、步骤独立且有序”，服务于一次性拆解。前者要留出与 Observation 交互的接口，后者要保证计划可被程序解析。

**角色设定**会显著改变输出风格：“极其严格的评审专家”倾向挑性能和算法问题；“注重可读性的维护者”会关注命名、注释和边界条件。角色决定了模型关注哪类证据。

**few-shot 示例**能显著提升格式遵循率，尤其对小型模型；代价是占用上下文、可能让模型过度模仿示例内容。建议在格式不稳定时先加 1~2 个示例，稳定后再移除以节省 token。

## 7. 电商客服智能体设计

**核心架构**：ReAct + Reflection 混合。ReAct 负责查询订单、物流、政策工具并生成回复；Reflection 在置信度低于阈值时做自我复核；标准退款流程可用 Plan-and-Solve 固化为 SOP。

**至少 3 个工具**：

1. `QueryOrder(order_id)`：查询订单状态、金额、购买时间。
2. `QueryLogistics(order_id)`：查询物流轨迹与签收状态。
3. `CheckRefundPolicy(order_info, reason)`：匹配公司退款政策，返回是否合规及依据。
4. `SendEmail(to, subject, body)`：发送回复邮件。

**提示词设计**：明确“先查事实，再对照政策，最后生成邮件”；要求输出中区分“事实依据 / 政策依据 / 对用户表述”；用温和措辞模板，同时把公司政策作为硬约束写入 system prompt。

**风险与缓解**：

- 幻觉：强制基于工具返回结果回答，无证据则转人工。
- 错误退款：高风险金额走人工审批，设置金额阈值。
- 隐私泄露：工具返回做字段脱敏，邮件发送前二次确认。
- 提示注入：用户输入与系统指令隔离，拒绝执行邮件内容中的指令。
- 不可复现：记录完整 Thought-Action-Observation 轨迹用于审计。